# TechLIZA

## Fine-Tuning a Small Language Model for Technology Interviews

TechLIZA is a technology media interviewer built by adapting
`Qwen/Qwen2.5-0.5B-Instruct` with LoRA supervised fine-tuning. Given the
previous interviewer question and the interviewee's latest response, the
model generates one concise follow-up question.

| Field | Details |
|---|---|
| Course | COMS2702: AI in Context |
| Institution | Columbia University |
| Term | Fall 2026 |
| Project Type | Individual Coursework |
| Status | Refactored |

**Research question:** Does fine-tuning a small language model on real
technology media interviews improve its ability to generate relevant
follow-up questions, and do those improvements generalize to interviews
from a different source?

This refactored notebook preserves the original project workflow and
reported results. Reusable functions are imported from `src/techliza` so
the notebook and Streamlit application do not maintain duplicate logic.

## How to Run This Notebook

Start Jupyter from the repository root and open this file. Install the
notebook dependencies first:

```powershell
python -m pip install -r requirements-notebook.txt
```

The selected LoRA adapter is already included. By default, the notebook
loads that adapter without rebuilding datasets or repeating training.
Enable the experiment flags below only when reproducing the corresponding
expensive stage. The full training workflow requires a CUDA GPU.

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

current_directory = Path.cwd().resolve()
REPOSITORY_ROOT = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.techliza.data import (
    build_episode_index,
    clean_follow_up_examples,
    extract_follow_up_examples,
    extract_two_turn_candidates,
    filter_technology_episodes,
    match_two_turn_context,
    split_by_episode,
)
from src.techliza.evaluation import (
    build_anthropic_examples,
    create_blind_options,
    remove_interview_boundaries,
    reveal_winners,
)
from src.techliza.model import (
    BASE_MODEL_NAME,
    build_interview_messages,
    generate_question,
    generate_with_model,
    load_techliza_model,
)
from src.techliza.prompts import OPENING_QUESTION, TRAINING_SYSTEM_PROMPT

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

ADAPTER_PATH = REPOSITORY_ROOT / "models" / "techliza_lora_final"
RESULTS_DIR = REPOSITORY_ROOT / "results"
WORK_DIR = REPOSITORY_ROOT / "outputs"

RUN_DATA_PIPELINE = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_CONTEXT_EXPERIMENT = False
RUN_CLI_DEMO = False
START_STREAMLIT = False
START_CLOUDFLARE_TUNNEL = False

print("Repository root:", REPOSITORY_ROOT)
print("CUDA available:", torch.cuda.is_available())
print("Submitted adapter:", ADAPTER_PATH)

## 1. Base Model Baseline

The original project first confirmed that the base Qwen model could
follow the interviewer prompt. Ten fixed cases were then used to document
baseline behavior before fine-tuning. Deterministic decoding was used for
all model comparisons.

In [ ]:
baseline_test_cases = [
    "AI coding assistants save me a lot of time when writing boilerplate code.",
    (
        "Open-source models matter because they reduce cost, improve "
        "transparency, and allow researchers to modify the architecture."
    ),
    (
        "Transformer models became dominant partly because parallel training "
        "made it practical to scale them much further than recurrent networks."
    ),
    (
        "I initially avoided using AI in my research because I worried about "
        "hallucinations, but recently I started using it for brainstorming "
        "rather than factual retrieval."
    ),
    (
        "I don't think larger models alone will lead to AGI. The limitation "
        "may be in the training paradigm rather than scale."
    ),
    (
        "Smaller startups benefit from open-source models because they cannot "
        "afford the compute required to train frontier models."
    ),
    "AI systems are becoming more capable, but evaluation has not improved at the same pace.",
    (
        "One of the biggest problems with AI in healthcare is that models may "
        "perform well overall while failing badly for certain patient groups."
    ),
    (
        "I think regulation should focus more on how AI systems are used than "
        "on the models themselves."
    ),
    (
        "The most useful thing AI has changed for me is not automation. It is "
        "that I can explore ideas much faster before deciding which ones are "
        "worth pursuing."
    ),
]

pd.DataFrame(
    {
        "case_id": range(1, len(baseline_test_cases) + 1),
        "interviewee_response": baseline_test_cases,
    }
)

## 2. NPR Data Acquisition and Inspection

The training data comes from the NPR Interview 2P corpus distributed
through ConvoKit. The original corpus contained 22,149 episodes, 22,267
speakers, and 428,624 utterances. Raw and processed interview text is not
committed to this repository.

In [ ]:
if RUN_DATA_PIPELINE:
    from convokit import Corpus, download

    npr_corpus_path = Path(download("npr-2p-corpus"))
    npr_corpus = Corpus(filename=str(npr_corpus_path))

    with (npr_corpus_path / "conversations.json").open(encoding="utf-8") as stream:
        conversations_data = json.load(stream)

    print("Conversations:", len(list(npr_corpus.iter_conversations())))
    print("Speakers:", len(list(npr_corpus.iter_speakers())))
    print("Utterances:", len(list(npr_corpus.iter_utterances())))
else:
    print("Data acquisition skipped. Set RUN_DATA_PIPELINE=True to reproduce it.")

## 3. NPR Data Preparation

The preparation sequence is unchanged from the original project:

1. Build an episode-level index.
2. Apply high-precision technology title patterns and exclusions.
3. Require at least eight utterances per episode.
4. Merge consecutive turns from the same speaker role.
5. Extract host–guest–host follow-up triples.
6. Apply the original word-count and one-question filters.
7. Split at the episode level using random seed 42.

In [ ]:
if RUN_DATA_PIPELINE:
    episode_df, episodes = build_episode_index(npr_corpus, conversations_data)
    technology_episode_df = filter_technology_episodes(episode_df)
    candidate_df = extract_follow_up_examples(
        technology_episode_df["episode_id"],
        episodes,
    )
    clean_training_df = clean_follow_up_examples(candidate_df)
    train_df, validation_df, test_df = split_by_episode(
        clean_training_df,
        random_state=RANDOM_SEED,
    )

    summary = pd.Series(
        {
            "all_episodes": len(episode_df),
            "technology_episodes": len(technology_episode_df),
            "candidate_examples": len(candidate_df),
            "clean_examples": len(clean_training_df),
            "train_examples": len(train_df),
            "validation_examples": len(validation_df),
            "test_examples": len(test_df),
        }
    )
    display(summary.to_frame("count"))
else:
    display(
        pd.Series(
            {
                "all_episodes": 22149,
                "broad_topic_candidates": 1440,
                "high_precision_candidates": 238,
                "technology_episodes": 220,
                "candidate_examples": 961,
                "clean_examples": 551,
                "train_examples": 436,
                "validation_examples": 56,
                "test_examples": 59,
            }
        ).to_frame("recorded_original_count")
    )

## 4. Prompt-Completion Datasets

Each example contains a system instruction, the previous interviewer
question, the interviewee response, and the next interviewer question as
the completion target. The reusable conversion function is shared by the
one-turn and controlled two-turn experiments.

In [ ]:
if RUN_DATA_PIPELINE:
    from src.techliza.training import dataframe_to_sft_dataset

    train_dataset = dataframe_to_sft_dataset(train_df)
    validation_dataset = dataframe_to_sft_dataset(validation_df)
    print("Training examples:", len(train_dataset))
    print("Validation examples:", len(validation_dataset))
    display(train_dataset[0])
else:
    print("Dataset conversion skipped with the data pipeline.")

## 5. Run 1: LoRA Fine-Tuning

The final coursework run used LoRA rank 16, alpha 32, dropout 0.05,
attention projection targets, three epochs, learning rate `2e-4`, FP16,
gradient accumulation of four, maximum sequence length 512, and seed 42.

Training is disabled by default because the selected adapter is committed
with the repository. Set both `RUN_DATA_PIPELINE` and `RUN_TRAINING` to
`True` to repeat this stage on a CUDA GPU.

In [ ]:
if RUN_TRAINING:
    if not RUN_DATA_PIPELINE:
        raise RuntimeError("RUN_TRAINING requires RUN_DATA_PIPELINE=True")
    if not torch.cuda.is_available():
        raise RuntimeError("The original FP16 training workflow requires a CUDA GPU")

    from peft import get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer
    from src.techliza.training import build_lora_config, build_sft_config

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    base_training_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype="auto",
        device_map="auto",
    )
    training_model = get_peft_model(base_training_model, build_lora_config())
    trainer = SFTTrainer(
        model=training_model,
        args=build_sft_config(str(WORK_DIR / "run1_training")),
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
    )
    train_result = trainer.train()
    reproduced_adapter_path = WORK_DIR / "run1_adapter"
    training_model.save_pretrained(reproduced_adapter_path)
    print("Reproduced adapter saved to:", reproduced_adapter_path)
else:
    print("Training skipped; the submitted adapter will be loaded below.")

In [ ]:
training_metrics = pd.read_csv(RESULTS_DIR / "metrics" / "training_metrics.csv")
display(training_metrics)

The final one-turn run's training loss decreased from 2.8341 to 2.6181.
Validation loss remained nearly flat and ended slightly higher, so output
inspection and held-out comparisons were used alongside loss.

## 6. Load the Submitted Adapter

This is the default execution path. The tokenizer and public base model
are downloaded from Hugging Face, and the repository's LoRA adapter is
attached without retraining.

In [ ]:
tokenizer, techliza_model = load_techliza_model(ADAPTER_PATH)
print("Loaded base model:", BASE_MODEL_NAME)
print("Loaded adapter:", ADAPTER_PATH)
print("Model device:", techliza_model.device)

In [ ]:
smoke_topic = "Open-source language models"
smoke_response = (
    "Open-source models help smaller research groups inspect model behavior "
    "and adapt systems without training from scratch."
)
smoke_question = generate_question(
    tokenizer,
    techliza_model,
    main_topic=smoke_topic,
    previous_question=OPENING_QUESTION,
    interviewee_response=smoke_response,
)
print("Interviewee:", smoke_response)
print("TechLIZA:", smoke_question)

## 7. Base Model and TechLIZA Comparisons

The original project compared the clean base model and the final adapter
with identical prompts and deterministic decoding. Enable
`RUN_EVALUATION` to regenerate outputs for the ten fixed cases.

In [ ]:
if RUN_EVALUATION:
    from transformers import AutoModelForCausalLM

    evaluation_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=evaluation_dtype,
        device_map="auto",
    )
    base_model.eval()

    comparison_rows = []
    for case_id, response in enumerate(baseline_test_cases, start=1):
        messages = [
            {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
            {"role": "user", "content": response},
        ]
        comparison_rows.append(
            {
                "case_id": case_id,
                "interviewee_response": response,
                "base_model_output": generate_with_model(
                    tokenizer, base_model, messages, max_new_tokens=100
                ),
                "techliza_output": generate_with_model(
                    tokenizer, techliza_model, messages, max_new_tokens=100
                ),
            }
        )
    fixed_case_comparison = pd.DataFrame(comparison_rows)
    display(fixed_case_comparison)
else:
    print("Model comparison skipped. Set RUN_EVALUATION=True to regenerate it.")

Fine-tuning made the model more likely to focus on concrete terms from the
response. The change was inconsistent: TechLIZA could still ask generic
questions, overuse clarification, repeat earlier material, or introduce a
weak new direction.

## 8. Training Data Diagnostics

The original analysis found that only 5 of 551 targets (0.9%) matched the
selected clarification patterns. Target questions averaged 23.0 words,
and 119 examples (21.6%) had no content-word overlap between the available
one-turn context and target question. This motivated the controlled
context experiment without changing the final Run 1 data.

In [ ]:
if RUN_DATA_PIPELINE:
    stopwords = {
        "the", "a", "an", "and", "or", "but", "if", "so", "to", "of",
        "in", "on", "for", "with", "at", "by", "from", "is", "are",
        "was", "were", "be", "been", "being", "do", "does", "did",
        "have", "has", "had", "you", "your", "i", "we", "they", "he",
        "she", "it", "this", "that", "these", "those", "what", "how",
        "why", "when", "where", "who", "which",
    }

    def content_words(text):
        import re
        words = re.findall(r"[a-zA-Z']+", str(text).lower())
        return {word for word in words if word not in stopwords and len(word) > 2}

    diagnostic_df = clean_training_df.copy()
    diagnostic_df["context_target_overlap"] = diagnostic_df.apply(
        lambda row: (
            len(
                content_words(row["previous_question"] + " " + row["interviewee_answer"])
                & content_words(row["next_question"])
            )
            / max(len(content_words(row["next_question"])), 1)
        ),
        axis=1,
    )
    display(diagnostic_df["context_target_overlap"].describe())
    print(
        "Zero-overlap targets:",
        int((diagnostic_df["context_target_overlap"] == 0).sum()),
    )
else:
    print("Diagnostics skipped with the data pipeline.")

## 9. Controlled One-Turn vs. Two-Turn Experiment

Only examples with valid earlier context were retained. The matched data
contained 198 training, 28 validation, and 25 test examples. Both models
used the same targets, LoRA configuration, training settings, split, and
seed; the only intended difference was the extra question-answer turn.

In [ ]:
if RUN_CONTEXT_EXPERIMENT:
    if not RUN_DATA_PIPELINE:
        raise RuntimeError(
            "RUN_CONTEXT_EXPERIMENT requires RUN_DATA_PIPELINE=True"
        )
    if not torch.cuda.is_available():
        raise RuntimeError("The original FP16 training workflow requires a CUDA GPU")

    from peft import get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer
    from src.techliza.training import (
        build_lora_config,
        build_sft_config,
        dataframe_to_sft_dataset,
    )

    two_turn_candidates = extract_two_turn_candidates(
        clean_training_df["episode_id"].unique(),
        episodes,
    )
    matched_two_turn = match_two_turn_context(
        clean_training_df,
        two_turn_candidates,
    )
    train_ids = set(train_df["episode_id"])
    validation_ids = set(validation_df["episode_id"])
    test_ids = set(test_df["episode_id"])
    run2_frames = {
        "train": matched_two_turn[matched_two_turn["episode_id"].isin(train_ids)],
        "validation": matched_two_turn[
            matched_two_turn["episode_id"].isin(validation_ids)
        ],
        "test": matched_two_turn[matched_two_turn["episode_id"].isin(test_ids)],
    }

    context_adapters = {}
    context_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    for label, include_earlier_turn in {
        "one_turn": False,
        "two_turn": True,
    }.items():
        training_data = dataframe_to_sft_dataset(
            run2_frames["train"], include_earlier_turn
        )
        validation_data = dataframe_to_sft_dataset(
            run2_frames["validation"], include_earlier_turn
        )
        fresh_base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype="auto",
            device_map="auto",
        )
        context_model = get_peft_model(fresh_base, build_lora_config())
        context_trainer = SFTTrainer(
            model=context_model,
            args=build_sft_config(str(WORK_DIR / f"run2_{label}")),
            train_dataset=training_data,
            eval_dataset=validation_data,
            processing_class=context_tokenizer,
        )
        context_trainer.train()
        output_path = WORK_DIR / f"run2_{label}_adapter"
        context_model.save_pretrained(output_path)
        context_adapters[label] = output_path
        del context_model, context_trainer, fresh_base
        torch.cuda.empty_cache()

    print("Controlled context adapters:", context_adapters)
else:
    print(
        "Context training skipped. Set RUN_DATA_PIPELINE=True and "
        "RUN_CONTEXT_EXPERIMENT=True to reproduce it."
    )

In [ ]:
evaluation_summary = pd.read_csv(
    RESULTS_DIR / "tables" / "evaluation_summary.csv"
)
display(evaluation_summary)

The two-turn model's final validation loss was lower, but the blinded
25-case comparison did not show a clear qualitative advantage. The
original one-turn adapter was retained because it used 436 training
examples rather than the matched experiment's 198.

## 10. Cross-Source Evaluation

The Anthropic Interviewer dataset is used only for evaluation. The
original workflow parses interviewer/interviewee turns, removes the first
onboarding and final closing example from each transcript, samples ten
examples per split, and then creates a ten-case set containing three
creative, three scientific, and four workforce examples.

In [ ]:
if RUN_EVALUATION:
    from datasets import load_dataset

    anthropic_dataset = load_dataset("Anthropic/AnthropicInterviewer")
    anthropic_examples = build_anthropic_examples(
        anthropic_dataset,
        ["workforce", "creatives", "scientists"],
    )
    anthropic_core = remove_interview_boundaries(anthropic_examples)
    balanced_sample = (
        anthropic_core.groupby("split", group_keys=False)
        .sample(n=10, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )
    anthropic_ten = pd.concat(
        [
            balanced_sample[balanced_sample["split"] == "creatives"].sample(
                n=3, random_state=RANDOM_SEED
            ),
            balanced_sample[balanced_sample["split"] == "scientists"].sample(
                n=3, random_state=RANDOM_SEED
            ),
            balanced_sample[balanced_sample["split"] == "workforce"].sample(
                n=4, random_state=RANDOM_SEED
            ),
        ],
        ignore_index=True,
    )

    cross_source_rows = []
    for case_id, row in anthropic_ten.iterrows():
        messages = [
            {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Previous interviewer question:\n{row['previous_question']}\n\n"
                    f"Interviewee response:\n{row['interviewee_answer']}"
                ),
            },
        ]
        cross_source_rows.append(
            {
                "case_id": case_id + 1,
                "split": row["split"],
                "previous_question": row["previous_question"],
                "interviewee_answer": row["interviewee_answer"],
                "reference_question": row["next_question"],
                "base_output": generate_with_model(
                    tokenizer, base_model, messages, max_new_tokens=100
                ),
                "techliza_output": generate_with_model(
                    tokenizer, techliza_model, messages, max_new_tokens=100
                ),
            }
        )
    cross_source_results = pd.DataFrame(cross_source_rows)
    blind_table, blind_mapping = create_blind_options(
        cross_source_results,
        "base_output",
        "techliza_output",
        "Base",
        "TechLIZA",
        random_state=RANDOM_SEED,
    )
    display(blind_table)
    print(
        "Record A, B, Tie, or Neither in a winner column before calling "
        "reveal_winners(blind_table, blind_mapping)."
    )
else:
    print("Cross-source generation skipped with RUN_EVALUATION=False.")

In the recorded ten-case blinded comparison, TechLIZA was preferred seven
times and the base model three times. This is preliminary evidence rather
than a stable performance estimate because the sample was small and used
one human reviewer.

## 11. Interactive Command-Line Test

This optional loop uses the same prompt and generation function as the
Streamlit application.

In [ ]:
if RUN_CLI_DEMO:
    print("TechLIZA:", OPENING_QUESTION)
    main_topic = input("You: ").strip()
    previous_question = OPENING_QUESTION
    interviewee_response = main_topic

    while interviewee_response.lower() != "quit":
        next_question = generate_question(
            tokenizer,
            techliza_model,
            main_topic,
            previous_question,
            interviewee_response,
        )
        print("TechLIZA:", next_question)
        previous_question = next_question
        interviewee_response = input("You: ").strip()
else:
    print("CLI demo disabled. Set RUN_CLI_DEMO=True to start it.")

## 12. Run the Streamlit Application

The notebook launches the repository's single `app.py` implementation;
it does not write or maintain a duplicate application. Set
`START_STREAMLIT=True` in the configuration cell, run the following cell,
and open `http://localhost:8501`.

In [ ]:
streamlit_process = None
if START_STREAMLIT:
    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(REPOSITORY_ROOT / "app.py"),
            "--server.port",
            "8501",
            "--server.address",
            "0.0.0.0",
        ]
    )
    print("Streamlit started at http://localhost:8501")
else:
    print("Streamlit disabled. Set START_STREAMLIT=True to launch it.")

In Google Colab, the Streamlit server requires a tunnel because Colab does
not expose localhost directly. Install the `cloudflared` executable in the
runtime, set `START_CLOUDFLARE_TUNNEL=True`, and run the next cell. The URL
is temporary and should not be treated as a deployment.

In [ ]:
cloudflare_process = None
if START_CLOUDFLARE_TUNNEL:
    if not START_STREAMLIT:
        raise RuntimeError("Start Streamlit before opening a tunnel")
    cloudflare_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8501"]
    )
    print("Cloudflare tunnel requested. Read the process output for its URL.")
else:
    print("Cloudflare tunnel disabled for local execution.")

## Limitations

- The training set is small and comes from one media source.
- The blind evaluations contain 25 and 10 cases and use one reviewer.
- External samples drawn from the same transcript may be correlated.
- The model can produce generic questions, repetition, unnecessary
  clarification, or topic drift.
- Exact training reproduction may vary with GPU hardware and library
  versions.
- A temporary Cloudflare tunnel is suitable for a classroom demonstration,
  not persistent deployment.

## Conclusion

LoRA fine-tuning changed the interviewing behavior of the 0.5-billion-
parameter base model while training approximately 0.44% of its parameters.
TechLIZA more often focused on concrete details, although the improvement
was inconsistent. Adding a second context turn lowered validation loss but
did not produce a clear human preference advantage, so the larger one-turn
adapter remained the final model. The cross-source result supports further
study with larger samples and more rigorous evaluation rather than a claim
that the system is complete.